# 모델 성능 비교 시각화 — 2026-05-04

**담당:** 경이 (kyeongyi)  
**목적:** Simple(베이스라인) vs KcELECTRA(파인튜닝) 성능 비교 차트 4종 생성

**GPU 불필요** — JSON 결과 파일만 있으면 실행 가능.

## 실행 전 필수 파일
| 파일 | 생성 방법 |
|------|-----------|
| `data/eval_results_simple_20260504.json` | `python scripts/evaluate_compare_v2_20260504.py` |
| `data/eval_results_kcelectra_20260504.json` | Colab에서 `03_train_kcelectra_v2_20260504.ipynb` 실행 후 복사 |

## 생성 파일 (타임스탬프: 20260504)
```
data/
├── compare_macro_f1_20260504.png
├── compare_per_class_f1_20260504.png
├── compare_confusion_matrix_20260504.png
├── compare_radar_f1_20260504.png
└── eval_comparison_summary_20260504.csv
```

In [ ]:
# ── 셀 1: 패키지 설치 (Colab 실행 시 주석 해제) ───────────────────
# !pip install matplotlib seaborn pandas -q

In [ ]:
# ── 셀 2: 임포트 + 한글 폰트 설정 ────────────────────────────────
import json
import platform
from pathlib import Path

import matplotlib
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import numpy as np
import pandas as pd
import seaborn as sns

# OS별 한글 폰트 (깨짐 방지)
if platform.system() == "Windows":
    matplotlib.rc("font", family="Malgun Gothic")
elif platform.system() == "Darwin":
    matplotlib.rc("font", family="AppleGothic")
else:  # Linux / Colab
    try:
        import subprocess
        subprocess.run(["apt-get", "install", "-y", "fonts-nanum"], capture_output=True)
        fm.fontManager.__init__()
        matplotlib.rc("font", family="NanumGothic")
    except Exception:
        pass

matplotlib.rcParams["axes.unicode_minus"] = False

# 경로 설정
import os
_IN_COLAB = os.path.exists("/content")
if _IN_COLAB:
    BASE_DIR = Path("/content")
else:
    BASE_DIR = Path(".").resolve().parent / "data"  # notebooks/ → data/

TS    = "20260504"   # 타임스탬프
LABELS = ["일정", "준비물", "제출", "비용", "건강·안전", "기타"]

print(f"데이터 경로: {BASE_DIR}")
print(f"플랫폼: {platform.system()}")

In [ ]:
# ── 셀 3: JSON 결과 로드 ──────────────────────────────────────────

def load_json(path: Path) -> dict:
    if not path.exists():
        print(f"[없음] {path.name}")
        return {}
    with open(path, encoding="utf-8") as f:
        return json.load(f)

simple_res    = load_json(BASE_DIR / f"eval_results_simple_{TS}.json")
kcelectra_res = load_json(BASE_DIR / f"eval_results_kcelectra_{TS}.json")

print(f"Simple    Macro F1: {simple_res.get('macro_f1', 'N/A')}")
print(f"KcELECTRA Macro F1: {kcelectra_res.get('macro_f1', 'N/A')}")

if not simple_res:
    print("[오류] eval_results_simple_20260504.json 없음 — evaluate_compare_v2_20260504.py 먼저 실행")
if not kcelectra_res:
    print("[안내] eval_results_kcelectra_20260504.json 없음 — 03_train_kcelectra_v2 Colab 실행 후 복사")
    print("       KcELECTRA 없이도 Simple 단독 차트는 생성됩니다.")

In [ ]:
# ── 셀 4: 차트 설정 헬퍼 ─────────────────────────────────────────

COLORS = {
    "simple":    "#4C72B0",   # 파란색 — 베이스라인
    "kcelectra": "#DD8452",   # 주황색 — 파인튜닝
}

def save_fig(fig, name: str):
    path = BASE_DIR / f"{name}_{TS}.png"
    fig.savefig(path, dpi=150, bbox_inches="tight")
    print(f"[저장] {path.name}")

In [ ]:
# ── 셀 5: 차트 1 — Macro F1 막대 그래프 ──────────────────────────
# 두 모델의 전체 성능을 한눈에 비교.
# 채택 기준선(Simple + 5%)을 빨간 점선으로 표시.

fig, ax = plt.subplots(figsize=(7, 5))

models = ["Simple\n(TF-IDF + LR)"]
values = [simple_res.get("macro_f1", 0)]
colors = [COLORS["simple"]]

if kcelectra_res:
    models.append("KcELECTRA\n(파인튜닝)")
    values.append(kcelectra_res.get("macro_f1", 0))
    colors.append(COLORS["kcelectra"])

bars = ax.bar(models, values, color=colors, edgecolor="white", linewidth=1.5, width=0.5)

# 막대 위 수치 표시
for bar, val in zip(bars, values):
    ax.text(bar.get_x() + bar.get_width()/2, val + 0.01,
            f"{val:.4f}", ha="center", va="bottom", fontsize=13, fontweight="bold")

# 채택 기준선 (Simple + 5%)
if simple_res:
    threshold = simple_res["macro_f1"] + 0.05
    ax.axhline(y=threshold, color="crimson", linestyle="--", linewidth=1.5,
               label=f"채택 기준 (Simple + 5%) = {threshold:.4f}")
    ax.legend(fontsize=10)

ax.set_ylim(0, 1.05)
ax.set_ylabel("Macro F1", fontsize=12)
ax.set_title("모델 성능 비교: Macro F1", fontsize=14, fontweight="bold")
ax.yaxis.grid(True, alpha=0.3)
ax.set_axisbelow(True)

plt.tight_layout()
save_fig(fig, "compare_macro_f1")
plt.show()

In [ ]:
# ── 셀 6: 차트 2 — 카테고리별 F1 막대 그래프 ──────────────────────
# 어떤 카테고리에서 어떤 모델이 더 강한지 보여줌.
# 취약 카테고리 발견 + 개선 방향 도출에 활용.

fig, ax = plt.subplots(figsize=(10, 5))

x = np.arange(len(LABELS))
width = 0.35

simple_f1s = [
    simple_res.get("per_class", {}).get(lbl, {}).get("f1", 0)
    for lbl in LABELS
]

bars1 = ax.bar(x - width/2, simple_f1s, width, label="Simple (베이스라인)",
               color=COLORS["simple"], edgecolor="white", linewidth=1)

if kcelectra_res:
    kc_f1s = [
        kcelectra_res.get("per_class", {}).get(lbl, {}).get("f1", 0)
        for lbl in LABELS
    ]
    bars2 = ax.bar(x + width/2, kc_f1s, width, label="KcELECTRA (파인튜닝)",
                   color=COLORS["kcelectra"], edgecolor="white", linewidth=1)

    for bar, val in zip(bars2, kc_f1s):
        ax.text(bar.get_x() + bar.get_width()/2, val + 0.01,
                f"{val:.2f}", ha="center", va="bottom", fontsize=9)

for bar, val in zip(bars1, simple_f1s):
    ax.text(bar.get_x() + bar.get_width()/2, val + 0.01,
            f"{val:.2f}", ha="center", va="bottom", fontsize=9)

ax.set_xticks(x)
ax.set_xticklabels(LABELS, fontsize=11)
ax.set_ylim(0, 1.15)
ax.set_ylabel("F1 Score", fontsize=12)
ax.set_title("카테고리별 F1 비교", fontsize=14, fontweight="bold")
ax.legend(fontsize=11)
ax.yaxis.grid(True, alpha=0.3)
ax.set_axisbelow(True)

plt.tight_layout()
save_fig(fig, "compare_per_class_f1")
plt.show()

In [ ]:
# ── 셀 7: 차트 3 — Confusion Matrix 비교 ─────────────────────────
# 어떤 카테고리끼리 혼동되는지 시각화.
# 대각선이 진할수록 좋은 모델.

n_plots = 2 if kcelectra_res else 1
fig, axes = plt.subplots(1, n_plots, figsize=(7 * n_plots, 6))

if n_plots == 1:
    axes = [axes]

# Simple Confusion Matrix
if simple_res.get("confusion_matrix"):
    cm_simple = np.array(simple_res["confusion_matrix"])
    sns.heatmap(cm_simple, annot=True, fmt="d", cmap="Blues",
                xticklabels=LABELS, yticklabels=LABELS, ax=axes[0])
    axes[0].set_title(f"Simple\nMacro F1={simple_res['macro_f1']:.4f}", fontsize=13)
    axes[0].set_xlabel("예측")
    axes[0].set_ylabel("실제")

# KcELECTRA Confusion Matrix
if kcelectra_res and kcelectra_res.get("confusion_matrix"):
    cm_kc = np.array(kcelectra_res["confusion_matrix"])
    sns.heatmap(cm_kc, annot=True, fmt="d", cmap="Oranges",
                xticklabels=LABELS, yticklabels=LABELS, ax=axes[1])
    axes[1].set_title(f"KcELECTRA (파인튜닝)\nMacro F1={kcelectra_res['macro_f1']:.4f}", fontsize=13)
    axes[1].set_xlabel("예측")
    axes[1].set_ylabel("실제")

plt.suptitle("Confusion Matrix 비교", fontsize=15, fontweight="bold", y=1.02)
plt.tight_layout()
save_fig(fig, "compare_confusion_matrix")
plt.show()

In [ ]:
# ── 셀 8: 차트 4 — 레이더 차트 ───────────────────────────────────
# 6개 카테고리 F1을 육각형으로 시각화.
# 각 꼭짓점이 하나의 카테고리 — 넓을수록 균형 잡힌 성능.

from matplotlib.patches import FancyArrowPatch

fig, ax = plt.subplots(figsize=(7, 7), subplot_kw=dict(polar=True))

angles = np.linspace(0, 2 * np.pi, len(LABELS), endpoint=False).tolist()
angles += angles[:1]   # 닫힌 다각형을 위해 첫 값 반복

def get_f1s(res: dict) -> list:
    vals = [res.get("per_class", {}).get(lbl, {}).get("f1", 0) for lbl in LABELS]
    return vals + vals[:1]

# Simple
simple_f1s_radar = get_f1s(simple_res)
ax.plot(angles, simple_f1s_radar, color=COLORS["simple"], linewidth=2,
        linestyle="solid", label="Simple (베이스라인)")
ax.fill(angles, simple_f1s_radar, color=COLORS["simple"], alpha=0.15)

# KcELECTRA
if kcelectra_res:
    kc_f1s_radar = get_f1s(kcelectra_res)
    ax.plot(angles, kc_f1s_radar, color=COLORS["kcelectra"], linewidth=2,
            linestyle="solid", label="KcELECTRA (파인튜닝)")
    ax.fill(angles, kc_f1s_radar, color=COLORS["kcelectra"], alpha=0.15)

ax.set_thetagrids(np.degrees(angles[:-1]), LABELS, fontsize=12)
ax.set_ylim(0, 1)
ax.set_yticks([0.2, 0.4, 0.6, 0.8, 1.0])
ax.set_yticklabels(["0.2", "0.4", "0.6", "0.8", "1.0"], fontsize=9)
ax.set_title("카테고리별 F1 — 레이더 차트", fontsize=14, fontweight="bold", pad=20)
ax.legend(loc="upper right", bbox_to_anchor=(1.3, 1.1), fontsize=11)

plt.tight_layout()
save_fig(fig, "compare_radar_f1")
plt.show()

In [ ]:
# ── 셀 9: 성능 요약 테이블 + CSV 저장 ────────────────────────────

rows = []
for label, res in [("Simple (TF-IDF + LR)", simple_res), ("KcELECTRA (파인튜닝)", kcelectra_res)]:
    if not res:
        continue
    row = {
        "모델":         label,
        "Macro F1":    res.get("macro_f1", "-"),
        "Macro Prec": res.get("macro_precision", "-"),
        "Macro Rec":  res.get("macro_recall", "-"),
    }
    for lbl in LABELS:
        row[f"F1_{lbl}"] = res.get("per_class", {}).get(lbl, {}).get("f1", "-")
    rows.append(row)

summary_df = pd.DataFrame(rows)
display(summary_df)

summary_path = BASE_DIR / f"eval_comparison_summary_{TS}.csv"
summary_df.to_csv(summary_path, index=False, encoding="utf-8-sig")
print(f"[저장] {summary_path.name}")

# 채택 판정
if simple_res and kcelectra_res:
    delta = kcelectra_res["macro_f1"] - simple_res["macro_f1"]
    print(f"\n=== 채택 판정 ===")
    print(f"Simple    Macro F1: {simple_res['macro_f1']:.4f}")
    print(f"KcELECTRA Macro F1: {kcelectra_res['macro_f1']:.4f}")
    print(f"Delta: {delta:+.4f}")
    if delta >= 0.05:
        print("KcELECTRA 5%+ 향상 -> KcELECTRA 채택 권장!")
    elif delta >= 0:
        print("KcELECTRA 소폭 향상 -> 추가 데이터/튜닝 권장")
    else:
        print("Simple 유지 권장")